# NavState IMU EKF Example

This notebook demonstrates using the NavState-specific EKF (NavStateImuEKF) with synthetic IMU data generated from a simple scenario. We compare estimated yaw/pitch/roll (YPR), velocity (x,y,z), and position (x,y,z) against ground truth.

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/python/gtsam/examples/NavStateImuEKFExample.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install GTSAM from pip if running in Google Colab
try:
    import google.colab  # type: ignore
    %pip install --quiet gtsam-develop
except Exception:
    pass  # Not in Colab

In [ ]:
import numpy as np
import plotly.graph_objects as go
import pandas as pd

import gtsam
from gtsam import NavState, Rot3
from gtsam import ConstantTwistScenario, ScenarioRunner

# Utility helpers
def rot3_ypr_rad(R: Rot3):
    return np.degrees(R.ypr())

## Configure Scenario and EKF

In [ ]:
# --- Scenario: camera orbiting a fixed point ---
radius = 30.0
angular_velocity = np.pi  # rad/sec
w_b = np.array([0, 0, -angular_velocity])
v_n = np.array([radius * angular_velocity, 0, 0])

scenario = ConstantTwistScenario(w_b, v_n)


In [ ]:
# Simulation parameters
dt = 1.0 / 180.0 # 1 degree per step
T = 4.0   # total duration (s)
N = int(T / dt)
params = gtsam.PreintegrationParams.MakeSharedD(9.81)
runner = ScenarioRunner(scenario, params, dt, gtsam.imuBias.ConstantBias(np.zeros(3), np.zeros(3)))


## Simulate IMU and run EKF

In [ ]:
# Initialize EKF
X0: NavState = scenario.navState(0.0)
P0 = np.eye(9) * 1e-2
ekf = gtsam.NavStateImuEKF(X0, P0, params)

In [ ]:
times = np.linspace(0.0, T, N + 1)

# Storage
ypr_true, ypr_est = [], []
vel_true, vel_est = [], []
pos_true, pos_est = [], []

Xk = X0
for k, t in enumerate(times):
    # Ground truth state
    X_true: NavState = scenario.navState(t)
    ypr_true.append(rot3_ypr_rad(X_true.attitude()))
    vel_true.append(np.asarray(X_true.velocity()))
    pos_true.append(np.asarray(X_true.position()))

    if k == 0:
        # Store initial estimate before any prediction
        X_est = ekf.state()
        ypr_est.append(rot3_ypr_rad(X_est.attitude()))
        vel_est.append(np.asarray(X_est.velocity()))
        pos_est.append(np.asarray(X_est.position()))
        continue

    # Simulate noise-free IMU readings
    omega_meas = runner.actualAngularVelocity(t - dt)  # at start of interval
    acc_meas = runner.actualSpecificForce(t - dt)

    # EKF predict
    ekf.predict(omega_meas, acc_meas, dt)
    X_est = ekf.state()
    ypr_est.append(rot3_ypr_rad(X_est.attitude()))
    vel_est.append(np.asarray(X_est.velocity()))
    pos_est.append(np.asarray(X_est.position()))

# Convert to arrays
ypr_true = np.unwrap(np.vstack(ypr_true), axis=0)
ypr_est = np.unwrap(np.vstack(ypr_est), axis=0)
vel_true = np.vstack(vel_true)
vel_est = np.vstack(vel_est)
pos_true = np.vstack(pos_true)
pos_est = np.vstack(pos_est)

## Plot Yaw/Pitch/Roll (degrees)

In [ ]:
# Build DataFrame from computed arrays
df_ypr = pd.DataFrame({
    't': times,
    'yaw_true': ypr_true[:, 0], 'pitch_true': ypr_true[:, 1], 'roll_true': ypr_true[:, 2],
    'yaw_est':  ypr_est[:, 0],  'pitch_est':  ypr_est[:, 1],  'roll_est':  ypr_est[:, 2],
})

fig = go.Figure()
for comp in ['yaw', 'pitch', 'roll']:
    fig.add_scatter(x=df_ypr['t'], y=df_ypr[f'{comp}_true'], name=f'{comp} true', line=dict(dash='dash'))
    fig.add_scatter(x=df_ypr['t'], y=df_ypr[f'{comp}_est'],  name=f'{comp} est')
fig.update_layout(title='YPR Comparison (degrees)', xaxis_title='Time (s)', yaxis_title='Angle (deg)')
fig.show()

## Plot Velocity (m/s)

In [ ]:
df_vel = pd.DataFrame({
    't': times,
    'vx_true': vel_true[:, 0], 'vy_true': vel_true[:, 1], 'vz_true': vel_true[:, 2],
    'vx_est':  vel_est[:, 0],  'vy_est':  vel_est[:, 1],  'vz_est':  vel_est[:, 2],
})

fig = go.Figure()
for comp in ['x','y','z']:
    fig.add_scatter(x=df_vel['t'], y=df_vel[f'v{comp}_true'], name=f'v{comp} true', line=dict(dash='dash'))
    fig.add_scatter(x=df_vel['t'], y=df_vel[f'v{comp}_est'],  name=f'v{comp} est')
fig.update_layout(title='Velocity Comparison (m/s)', xaxis_title='Time (s)', yaxis_title='Velocity (m/s)')
fig.show()

## Plot Position (m)

In [ ]:
df_pos = pd.DataFrame({
    't': times,
    'x_true': pos_true[:, 0], 'y_true': pos_true[:, 1], 'z_true': pos_true[:, 2],
    'x_est':  pos_est[:, 0],  'y_est':  pos_est[:, 1],  'z_est':  pos_est[:, 2],
})

fig = go.Figure()
for comp in ['x','y','z']:
    fig.add_scatter(x=df_pos['t'], y=df_pos[f'{comp}_true'], name=f'{comp} true', line=dict(dash='dash'))
    fig.add_scatter(x=df_pos['t'], y=df_pos[f'{comp}_est'],  name=f'{comp} est')
fig.update_layout(title='Position Comparison (m)', xaxis_title='Time (s)', yaxis_title='Position (m)')
fig.show()